# 🤖 AI Intraday Scalping System — Google Colab
### NSE/BSE Markets · Angel One SmartAPI · Capital: ₹1,00,000

---

**Steps covered in this notebook:**
1. Install dependencies
2. Import modules & configure
3. Step 1: Stock Universe + Pre-market Filter
4. Step 2: AI Scoring Engine (7 parameters)
5. Step 3: Scalping Strategy Engine
6. Step 4: Paper Trade Simulation
7. Daily P&L Report
8. Multi-day Simulation & Performance Metrics
9. Angel One SmartAPI Live Mode (Optional)

> ⚠️ **Paper Trading Only** — No real orders are placed.

---
## Cell 1 — Install Dependencies

In [ ]:
# Install all required packages
!pip install -q yfinance pandas numpy ta streamlit smartapi-python textblob plotly pyotp loguru tabulate colorama scipy scikit-learn python-dotenv
print('✅ All packages installed successfully!')

---
## Cell 2 — Clone Repository & Import Modules

In [ ]:
import subprocess
import sys
import os

# Clone the repository (skip if already present)
REPO_URL = 'https://github.com/harnepal-hub/ai-scalping-system.git'
REPO_DIR = '/content/ai-scalping-system'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print(f'✅ Repository cloned to {REPO_DIR}')
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    print(f'✅ Repository updated at {REPO_DIR}')

# Add paths
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
os.chdir(REPO_DIR)

# Create log directory
os.makedirs('logs', exist_ok=True)
os.makedirs('data', exist_ok=True)

print('\n📁 Working directory:', os.getcwd())

In [ ]:
# Import all system modules
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from datetime import datetime, date

# Project modules
from config.config import (
    CAPITAL, MAX_POSITIONS, CAPITAL_PER_STOCK,
    MAX_RISK_PER_TRADE, MAX_DAILY_LOSS,
    MORNING_START, MORNING_END,
    AFTERNOON_START, AFTERNOON_END,
    ALL_SYMBOLS, WEIGHTS
)

from src.step1_universe_filter import (
    get_universe, get_premarket_data,
    apply_premarket_filters, get_filtered_universe
)
from src.step2_scoring_engine import (
    calculate_all_scores, get_top5,
    normalize_score, display_score_table
)
from src.step3_strategy_engine import (
    generate_trade_setup, check_entry_signal,
    display_trade_card, calculate_atr
)
from src.step4_paper_trader import (
    PaperTrader, calculate_charges
)

print('✅ All modules imported successfully!')
print(f'\n📊 Configuration:')
print(f'   Capital          : ₹{CAPITAL:,.0f}')
print(f'   Max positions    : {MAX_POSITIONS}')
print(f'   Capital / stock  : ₹{CAPITAL_PER_STOCK:,.0f}')
print(f'   Max risk / trade : ₹{MAX_RISK_PER_TRADE:,.0f}')
print(f'   Morning session  : {MORNING_START} – {MORNING_END}')
print(f'   Afternoon session: {AFTERNOON_START} – {AFTERNOON_END}')
print(f'   Universe size    : {len(ALL_SYMBOLS)} stocks')

---
## Cell 3 — Step 1: Load Universe & Pre-market Filter

In [ ]:
print('=' * 60)
print('STEP 1: STOCK UNIVERSE FILTER')
print('=' * 60)

# Get full universe
full_universe = get_universe()
print(f'\n📋 Full universe: {len(full_universe)} stocks')

# For Colab demo: use a smaller subset for faster execution
# Change to full_universe for production
DEMO_SUBSET = [
    'RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS', 'ICICIBANK.NS',
    'SBIN.NS', 'BHARTIARTL.NS', 'AXISBANK.NS', 'WIPRO.NS', 'HCLTECH.NS',
    'TATAMOTORS.NS', 'TATASTEEL.NS', 'NTPC.NS', 'POWERGRID.NS', 'ONGC.NS',
    'BAJFINANCE.NS', 'MARUTI.NS', 'TITAN.NS', 'SUNPHARMA.NS', 'DRREDDY.NS',
]

print(f'\n🔍 Running pre-market filter on {len(DEMO_SUBSET)} stocks (demo subset)...')
print('   (This fetches 20 days of daily data via yfinance — may take ~30s)\n')

# Fetch daily data for filtering
market_data = get_premarket_data(DEMO_SUBSET, period='20d', interval='1d')
filter_df = apply_premarket_filters(market_data)

if not filter_df.empty:
    passed_df = filter_df[filter_df['passes_filter']]
    tradable_symbols = passed_df['symbol'].tolist()

    print(f'\n✅ Tradable stocks after filter: {len(tradable_symbols)}')
    print(passed_df[['symbol', 'last_price', 'avg_volume', 'pct_change']].to_string(index=False))
else:
    print('⚠️  Filter returned no data. Using full demo subset as fallback.')
    tradable_symbols = DEMO_SUBSET

---
## Cell 4 — Step 2: AI Scoring Engine

In [ ]:
print('=' * 60)
print('STEP 2: AI SCORING ENGINE')
print('=' * 60)

print('\n📊 Fetching 5-min intraday data for scoring...')
print('   (This may take 1-2 minutes for multiple stocks)\n')

# Fetch 5-min intraday data
intraday_data = get_premarket_data(tradable_symbols, period='5d', interval='5m')
print(f'✅ Fetched intraday data for {len(intraday_data)} stocks')

# Score morning session
print('\n🌅 Scoring MORNING session...')
morning_scores = calculate_all_scores(
    tradable_symbols,
    intraday_data=intraday_data,
    session='morning'
)

if not morning_scores.empty:
    display_score_table(morning_scores, session='morning')
    morning_top5 = get_top5(morning_scores)
    print(f'\n🏆 MORNING TOP 5:')
    print(morning_top5[['rank', 'symbol', 'direction', 'composite_score']].to_string(index=False))

# Score afternoon session
print('\n🌆 Scoring AFTERNOON session...')
afternoon_scores = calculate_all_scores(
    tradable_symbols,
    intraday_data=intraday_data,
    session='afternoon'
)

if not afternoon_scores.empty:
    display_score_table(afternoon_scores, session='afternoon')
    afternoon_top5 = get_top5(afternoon_scores)
    print(f'\n🏆 AFTERNOON TOP 5:')
    print(afternoon_top5[['rank', 'symbol', 'direction', 'composite_score']].to_string(index=False))

---
## Cell 5 — Step 3: Strategy Engine — Trade Cards

In [ ]:
print('=' * 60)
print('STEP 3: SCALPING STRATEGY ENGINE')
print('=' * 60)

def generate_cards_for_top5(top5_df, session_name, intraday_data):
    """Generate and display trade cards for the top 5 scored stocks."""
    trade_setups = []

    for _, row in top5_df.iterrows():
        sym = row['symbol']
        direction = row['direction']

        # Get latest price and ATR
        df = intraday_data.get(sym)
        if df is None or df.empty:
            print(f'  ⚠️  No data for {sym}, skipping.')
            continue

        last_price = float(df['Close'].dropna().iloc[-1])
        atr = calculate_atr(df, period=14)

        # Check entry signal
        is_valid, reason = check_entry_signal(df, session=session_name, direction=direction)
        signal_icon = '✅' if is_valid else '⚠️'
        print(f'  {signal_icon} {sym.replace(".NS", ""):<15} Signal: {reason}')

        # Generate setup (regardless of signal for demo purposes)
        setup = generate_trade_setup(
            sym, last_price, atr, direction, session_name
        )
        setup['composite_score'] = row.get('composite_score', 0)
        trade_setups.append(setup)
        display_trade_card(setup)

    return trade_setups

# Morning trade cards
print('\n🌅 MORNING SESSION — TRADE SETUPS')
print('-' * 50)
morning_setups = []
if 'morning_top5' in dir() and not morning_top5.empty:
    morning_setups = generate_cards_for_top5(morning_top5, 'morning', intraday_data)

# Afternoon trade cards
print('\n🌆 AFTERNOON SESSION — TRADE SETUPS')
print('-' * 50)
afternoon_setups = []
if 'afternoon_top5' in dir() and not afternoon_top5.empty:
    afternoon_setups = generate_cards_for_top5(afternoon_top5, 'afternoon', intraday_data)

---
## Cell 6 — Step 4: Paper Trade Simulation

In [ ]:
print('=' * 60)
print('STEP 4: PAPER TRADE SIMULATION')
print('=' * 60)

# Initialise paper trader
trader = PaperTrader(capital=CAPITAL)

# Open trades based on generated setups
all_setups = morning_setups + afternoon_setups

if not all_setups:
    print('\n⚠️  No trade setups generated from live data.')
    print('    Using hardcoded demo trades instead...\n')

    # Demo trades for illustration
    demo_trades = [
        ('RELIANCE.NS', 'LONG',  2450.00, 2435.25, 2459.50, 2472.50, 8,  'morning'),
        ('HDFCBANK.NS', 'LONG',  1680.00, 1666.60, 1686.50, 1697.00, 11, 'morning'),
        ('TCS.NS',      'LONG',  3890.00, 3867.50, 3899.00, 3916.00, 5,  'afternoon'),
    ]
    for sym, dir_, entry, sl, tp1, tp2, qty, sess in demo_trades:
        trader.open_trade(sym, dir_, entry, sl, tp1, tp2, qty, sess)

    # Simulate price movements (demo)
    print('  Simulating price movements...')
    demo_exits = [
        ('RELIANCE.NS', 2460.00),   # TP1
        ('RELIANCE.NS', 2473.00),   # TP2
        ('HDFCBANK.NS', 1665.00),   # SL
        ('TCS.NS',      3921.00),   # TP2
    ]
    for sym, price in demo_exits:
        result = trader.update_prices(sym, price)
        if result:
            print(f'  {sym.replace(".NS","")}: {result} hit @ ₹{price:.2f}')
else:
    print(f'\n📥 Opening {len(all_setups)} paper trades...')

    for setup in all_setups:
        success = trader.open_trade(
            stock=setup['stock'],
            direction=setup['direction'],
            entry=setup['entry'],
            sl=setup['sl'],
            tp1=setup['tp1'],
            tp2=setup['tp2'],
            qty=setup['qty'],
            session=setup['session'],
            composite_score=setup.get('composite_score', 0),
            atr=setup.get('atr', 0),
            rr_ratio=setup.get('rr_ratio', 0),
        )
        sym_display = setup['stock'].replace('.NS', '')
        icon = '✅' if success else '❌'
        print(f'  {icon} {sym_display} {setup["direction"]} @ ₹{setup["entry"]:.2f}  Qty={setup["qty"]}')

    # Square off any remaining open trades (EOD)
    print('\n  ⏰ EOD square-off...')
    trader.eod_squareoff()

print(f'\n✅ Simulation complete. Total closed trades: {len(trader.closed_trades)}')

---
## Cell 7 — Daily P&L Report

In [ ]:
import plotly.graph_objects as go
from IPython.display import display

# Print formatted daily report
trader.print_daily_report()

# Save to CSV
TRADE_CSV = 'data/paper_trades.csv'
trader.save_to_csv(TRADE_CSV)
print(f'💾 Trades saved to {TRADE_CSV}')

# Plotly cumulative P&L chart
summary = trader.get_daily_summary()
if summary['trades']:
    trades_df = pd.DataFrame(summary['trades'])
    trades_df['cumulative_pnl'] = trades_df['net_pnl'].cumsum()

    fig = go.Figure()

    # Bar chart for individual trades
    colors = ['#00c853' if v >= 0 else '#d32f2f' for v in trades_df['net_pnl']]
    fig.add_trace(go.Bar(
        x=[t.replace('.NS', '') for t in trades_df['stock']],
        y=trades_df['net_pnl'],
        name='Net P&L per Trade',
        marker_color=colors,
        text=[f'₹{v:+.0f}' for v in trades_df['net_pnl']],
        textposition='outside',
    ))

    fig.update_layout(
        title='📊 Paper Trade Results — Net P&L per Trade',
        xaxis_title='Stock',
        yaxis_title='Net P&L (₹)',
        height=400,
        showlegend=False,
    )
    fig.show()

    # Cumulative P&L line
    fig2 = go.Figure(go.Scatter(
        x=list(range(len(trades_df))),
        y=CAPITAL + trades_df['cumulative_pnl'],
        mode='lines+markers',
        line=dict(color='#1976d2', width=2),
        fill='tozeroy',
        fillcolor='rgba(25, 118, 210, 0.1)',
        name='Capital',
    ))
    fig2.add_hline(y=CAPITAL, line_dash='dash', line_color='gray', annotation_text='Initial Capital')
    fig2.update_layout(
        title='📈 Cumulative Capital Curve',
        xaxis_title='Trade #',
        yaxis_title='Capital (₹)',
        height=400,
    )
    fig2.show()
else:
    print('No trades to chart.')

---
## Cell 8 — Multi-Day Simulation & Performance Metrics

In [ ]:
print('=' * 60)
print('MULTI-DAY SIMULATION (5 trading days — sample data)')
print('=' * 60)

# Load sample trade history (ships with the repo)
sample_csv = 'data/sample_trades.csv'

multi_trader = PaperTrader(capital=CAPITAL)
multi_trader.load_history(sample_csv)

if not multi_trader.closed_trades:
    print('\n⚠️  Sample trades not found. Generating synthetic data...')
    import random
    random.seed(42)
    np.random.seed(42)

    stocks = ['RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS', 'SBIN.NS']
    prices = [2450, 3890, 1680, 1590, 812]

    from datetime import timedelta
    base_date = date(2026, 3, 18)

    cap = float(CAPITAL)
    for day_offset in range(5):
        trade_date = base_date + timedelta(days=day_offset)
        for i, (sym, price) in enumerate(zip(stocks, prices)):
            entry = price * (1 + np.random.uniform(-0.005, 0.005))
            move = np.random.choice([-1, 1]) * np.random.uniform(0.003, 0.012) * entry
            exit_price = entry + move
            qty = max(1, int(20000 / entry))
            gross = (exit_price - entry) * qty
            fees = calculate_charges(min(entry, exit_price), max(entry, exit_price), qty)
            net = gross - fees
            cap += net

            multi_trader.closed_trades.append({
                'date': str(trade_date),
                'session': 'morning' if i < 3 else 'afternoon',
                'stock': sym,
                'symbol_token': '',
                'direction': 'LONG',
                'entry_price': round(entry, 2),
                'exit_price': round(exit_price, 2),
                'qty': qty,
                'sl': round(entry * 0.994, 2),
                'tp1': round(entry * 1.005, 2),
                'tp2': round(entry * 1.012, 2),
                'exit_reason': 'TP2' if gross > 0 else 'SL',
                'gross_pnl': round(gross, 2),
                'fees': round(fees, 2),
                'net_pnl': round(net, 2),
                'entry_time': f'{trade_date} 09:30:00',
                'exit_time': f'{trade_date} 10:15:00',
                'composite_score': round(np.random.uniform(0.55, 0.85), 4),
                'atr': round(price * 0.004, 2),
                'rr_ratio': 2.3,
                'capital_after': round(cap, 2),
            })
    multi_trader.capital = cap

all_trades_df = pd.DataFrame(multi_trader.closed_trades)
all_trades_df['net_pnl'] = pd.to_numeric(all_trades_df['net_pnl'], errors='coerce').fillna(0)
all_trades_df['date'] = pd.to_datetime(all_trades_df['date'])

# ── Performance Metrics ─────────────────────────────────────────────────────
total = len(all_trades_df)
wins = (all_trades_df['net_pnl'] >= 0).sum()
win_rate = wins / total * 100
gross_profit = all_trades_df[all_trades_df['net_pnl'] > 0]['net_pnl'].sum()
gross_loss = abs(all_trades_df[all_trades_df['net_pnl'] < 0]['net_pnl'].sum())
profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')

daily_pnl = all_trades_df.groupby(all_trades_df['date'].dt.date)['net_pnl'].sum()
daily_ret = daily_pnl / CAPITAL
sharpe = (daily_ret.mean() / daily_ret.std() * (250**0.5)) if daily_ret.std() > 0 else 0
cum_cap = CAPITAL + daily_pnl.cumsum()
peak = cum_cap.cummax()
max_dd = ((cum_cap - peak) / peak * 100).min()

print(f'\n{'='*50}')
print(f'  MULTI-DAY PERFORMANCE SUMMARY')
print(f'{'='*50}')
print(f'  Total trades    : {total}')
print(f'  Wins            : {wins} ({win_rate:.1f}%)')
print(f'  Losses          : {total - wins}')
print(f'  Profit Factor   : {profit_factor:.2f}')
print(f'  Sharpe Ratio    : {sharpe:.2f} (annualised)')
print(f'  Max Drawdown    : {max_dd:.2f}%')
print(f'  Net P&L         : ₹{all_trades_df["net_pnl"].sum():+,.2f}')
print(f'  Final Capital   : ₹{multi_trader.capital:,.2f}')
print(f'  Total Return    : {(multi_trader.capital - CAPITAL)/CAPITAL*100:+.2f}%')
print(f'{'='*50}')

# ── Charts ─────────────────────────────────────────────────────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Daily P&L', 'Cumulative Capital'),
    vertical_spacing=0.15
)

# Daily P&L bars
colors = ['#00c853' if v >= 0 else '#d32f2f' for v in daily_pnl]
fig.add_trace(go.Bar(
    x=[str(d) for d in daily_pnl.index],
    y=daily_pnl.values,
    marker_color=colors,
    name='Daily Net P&L',
    text=[f'₹{v:+,.0f}' for v in daily_pnl.values],
    textposition='outside',
), row=1, col=1)

# Capital curve
fig.add_trace(go.Scatter(
    x=[str(d) for d in cum_cap.index],
    y=cum_cap.values,
    mode='lines+markers',
    line=dict(color='#1976d2', width=2),
    fill='tozeroy',
    fillcolor='rgba(25, 118, 210, 0.1)',
    name='Capital',
), row=2, col=1)

fig.add_hline(y=CAPITAL, line_dash='dash', line_color='gray', row=2, col=1)
fig.update_layout(title='📈 Multi-Day Paper Trading Performance', height=700, showlegend=False)
fig.show()

---
## Cell 9 — Angel One SmartAPI Live Mode (Optional)

This cell shows how to connect to the Angel One API for live data.
**Paper trading only — no real orders are placed.**

### Setup Instructions
1. Get your API key from [Angel One SmartAPI](https://smartapi.angelbroking.com/)
2. Enable TOTP in your Angel One account
3. Add credentials to Colab Secrets (left sidebar → 🔑 icon):
   - `ANGEL_ONE_API_KEY`
   - `ANGEL_ONE_CLIENT_ID`
   - `ANGEL_ONE_PASSWORD`
   - `ANGEL_ONE_TOTP_SECRET`

In [ ]:
# ── Angel One SmartAPI Live Mode ────────────────────────────────────────────
# Set USE_LIVE_DATA = True to enable (requires valid API credentials)
USE_LIVE_DATA = False

if USE_LIVE_DATA:
    try:
        from google.colab import userdata
        import pyotp
        from SmartApi import SmartConnect

        API_KEY       = userdata.get('ANGEL_ONE_API_KEY')
        CLIENT_ID     = userdata.get('ANGEL_ONE_CLIENT_ID')
        PASSWORD      = userdata.get('2244')
        TOTP_SECRET   = userdata.get('ANGEL_ONE_TOTP_SECRET')

        # Generate TOTP
        totp = pyotp.TOTP(TOTP_SECRET).now()

        # Connect
        smartapi = SmartConnect(api_key=API_KEY)
        session_data = smartapi.generateSession(CLIENT_ID, PASSWORD, totp)

        if session_data.get('status'):
            auth_token = session_data['data']['jwtToken']
            refresh_token = session_data['data']['refreshToken']
            print(f'✅ Connected to Angel One SmartAPI')
            print(f'   Client ID: {CLIENT_ID}')

            # Example: Fetch live LTP for RELIANCE
            # ltp_data = smartapi.ltpData('NSE', 'RELIANCE-EQ', '2885')
            # print(f'   RELIANCE LTP: ₹{ltp_data["data"]["ltp"]}')

            print('\n💡 Angel One is connected. You can now:')
            print('   • Fetch live prices: smartapi.ltpData(exchange, symbol, token)')
            print('   • Stream real-time: use SmartWebSocket for tick data')
            print('   • All paper trades still use PaperTrader (no real orders)')
        else:
            print('❌ Angel One login failed:', session_data.get('message'))

    except ImportError:
        print('❌ SmartAPI not installed. Run: !pip install smartapi-python')
    except Exception as e:
        print(f'❌ Angel One connection error: {e}')
        print('   Check your credentials in Colab Secrets.')
else:
    print('ℹ️  Angel One live mode is DISABLED (USE_LIVE_DATA = False).')
    print('   Using yfinance for all data. Set USE_LIVE_DATA = True to enable.')
    print('\n   Required Colab Secrets:')
    print('   • ANGEL_ONE_API_KEY')
    print('   • ANGEL_ONE_CLIENT_ID')
    print('   • ANGEL_ONE_PASSWORD')
    print('   • ANGEL_ONE_TOTP_SECRET')

---
## 🎯 Summary

| Step | Module | Status |
|------|--------|--------|
| 1 | Universe Filter | ✅ Complete |
| 2 | AI Scoring Engine | ✅ Complete |
| 3 | Strategy Engine | ✅ Complete |
| 4 | Paper Trader + P&L | ✅ Complete |
| 5 | ML Model (coming) | 🔄 Planned |
| 6 | Live Execution (coming) | 🔄 Planned |

### ▶️ Run the Streamlit Dashboard
```bash
!streamlit run /content/ai-scalping-system/src/dashboard.py &
# Open the ngrok URL shown in output
```

---
> ⚠️ **Disclaimer:** This is a paper trading simulation. No real trades are executed. Not financial advice.

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 10 — LIVE NSE SCORING (Run anytime 9 AM–3 PM)
# ═══════════════════════════════════════════════════
import sys, os
sys.path.insert(0, '/content/ai-scalping-system')

from src.step1_universe_filter import UniverseFilter
from src.step2_scoring_engine import ScoringEngine
from src.step3_strategy_engine import StrategyEngine

# Step 1 — Filter Universe
print("🔍 Scanning NSE universe...")
uf = UniverseFilter()
filtered_stocks = uf.get_filtered_universe()
print(f"✅ {len(filtered_stocks)} stocks passed filter\n")

# Step 2 — Score & Get Top 5
print("🧠 Running AI Scoring Engine...")
se = ScoringEngine()

# Morning session picks
morning_scores = se.calculate_all_scores(filtered_stocks, session='morning')
top5_morning = se.get_top5(morning_scores)

print("\n🌅 TOP 5 MORNING PICKS:")
print("="*65)
for i, row in top5_morning.iterrows():
    print(f"  {row['rank']}. {row['symbol']:<15} Score: {row['composite_score']:.3f}  [{row['direction']}]")
print("="*65)

# Afternoon session picks
afternoon_scores = se.calculate_all_scores(filtered_stocks, session='afternoon')
top5_afternoon = se.get_top5(afternoon_scores)

print("\n☀️  TOP 5 AFTERNOON PICKS:")
print("="*65)
for i, row in top5_afternoon.iterrows():
    print(f"  {row['rank']}. {row['symbol']:<15} Score: {row['composite_score']:.3f}  [{row['direction']}]")
print("="*65)

# Step 3 — Generate Trade Cards for Morning Top 5
print("\n📋 GENERATING TRADE SETUPS...\n")
strategy = StrategyEngine()

for _, stock in top5_morning.iterrows():
    setup = strategy.generate_trade_setup(
        stock=stock['symbol'],
        price=stock['current_price'],
        atr=stock['atr'],
        direction=stock['direction'],
        session='morning'
    )
    strategy.display_trade_card(setup)